# 03 - Data Warehouse & Iceberg Cube

Notebook n?y t?o star schema cho PostgreSQL/SQL Server v? t?nh Iceberg Cube theo c?c ch? ?? kinh doanh.

In [1]:
from src.etl.extract import load_olist_tables
from src.etl.transform import build_order_features, build_star_schema

tables = load_olist_tables()
order_features = build_order_features(tables)
star_schema = build_star_schema(tables, order_features)
{name: df.shape for name, df in star_schema.items()}

{'dim_date': (634, 7), 'dim_customer': (99441, 5), 'dim_seller': (3095, 4), 'dim_product': (32951, 10), 'dim_payment': (38, 4), 'dim_order_status': (8, 1), 'fact_order_items': (112650, 16)}

In [2]:
# Uncomment khi ?? c?u h?nh .env v? t?o database olist_dwh trong PostgreSQL/SQL Server.
# from src.etl.load import load_star_schema_to_database
# load_star_schema_to_database(star_schema)

Warehouse load ???c th?c hi?n b?ng scripts/load_warehouse.py sau khi c?u h?nh .env.


In [3]:
from src.olap.iceberg_cube import compute_default_olist_cubes

cubes = compute_default_olist_cubes(order_features)
{name: cube.shape for name, cube in cubes.items()}

{'sales_by_time_category_state': (841, 11), 'delivery_quality': (432, 11), 'payment_satisfaction': (77, 11), 'geo_trade_lanes': (389, 11)}

In [4]:
cubes['delivery_quality'].sort_values(['bad_review_rate', 'count_orders'], ascending=False).head(20)

                                                        cuboid  dimension_count seller_state product_category_name_english is_delayed  count_orders  sum_revenue  avg_review_score  avg_delivery_days  delay_rate  bad_review_rate
13                                                seller_state                1      unknown                           ALL        ALL           775         0.00          1.721561                NaN         0.0         0.854701
189               seller_state x product_category_name_english                2      unknown                       unknown        ALL           775         0.00          1.721561                NaN         0.0         0.854701
210                                  seller_state x is_delayed                2      unknown                           ALL          0           775         0.00          1.721561                NaN         0.0         0.854701
431  seller_state x product_category_name_english x is_delayed                3      unknown